# Integrated PEH Pipeline Notebook

This notebook is the beginner-friendly entry point for the post-GRF pipeline.

Workflow:
1. use `periodic_grf_sdf.ipynb` to create a unit-cell dataset `.npz`
2. set that dataset path in the configuration cell below
3. run all cells here to build meshes, solve FEM, create the integrated dataset, and generate reports

If you do not already have a unit-cell dataset, run `periodic_grf_sdf.ipynb` first.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

PosixPath('/home/gijeong/Inverse Design')

## Configuration

- `SOURCE_UNIT_CELL_NPZ` should point to the dataset created by `periodic_grf_sdf.ipynb`
- If `LIMIT = 3` and `SAMPLE_IDS` is empty, the notebook requests 3 successful solid exports by rejection/resampling through the candidate dataset
- If `SAMPLE_IDS` is non-empty, it overrides the limit and runs those exact samples
- Leave the geometry scale fields as `None` to use the shared YAML problem specification
- `EXACT_CAD = True` preserves the generated topology and rejects disconnected tiled substrates
- `REPAIR_CAD = True` opts into explicit bridge geometry for disconnected samples while still exporting solid STEP bodies

- `CAD_REFERENCE_SIZE_SCALE` controls CAD feature-check precision independently of the in-house solver mesh
- keep `LIMIT_SOLVER_MESH_BY_THICKNESS = False` for thin plates unless you explicitly want the old, very fine 3D mesh behavior
- the default in-house solver now batches all samples in one Docker run, skips existing outputs, and uses quadratic solid interpolation for better bending accuracy


In [2]:
SOURCE_UNIT_CELL_NPZ = "data/unit_cell_dataset.npz"
RUN_NAME = "0409(5)"
LIMIT = 1
SAMPLE_IDS = "0"
MATERIALIZE_INPUT_DATASET = True
OUTPUT_ROOT = "runs"
DOCKER_IMAGE = "dolfinx/dolfinx:stable"
PROBLEM_SPEC_PATH = "configs/peh_inverse_design_spec.yaml"

SUBSTRATE_THICKNESS_M = 1.0e-3
PIEZO_THICKNESS_M = 1e-4
MESH_SIZE_SCALE = 0.08
CAD_REFERENCE_SIZE_SCALE = 0.01
LIMIT_SOLVER_MESH_BY_THICKNESS = False
SOLVER_MESH_BACKEND = "layered_tet"
SUBSTRATE_LAYERS = None
PIEZO_LAYERS = None
SOLVER_NUM_MODES = 8
SOLVER_SEARCH_POINTS = 301
SOLVER_ELEMENT_ORDER = 2
SOLVER_STORE_MODE_SHAPES = True
SKIP_EXISTING_SOLVER_OUTPUTS = True
STRICT_ANSYS_PARITY = True
PEAK_SEARCH_SEED = "f1"
AUDIT_ANSYS_MODAL_HZ = 0.58
AUDIT_ANSYS_FRF_PEAK_HZ = 0.58
AUDIT_ANSYS_VOLTAGE_V = 242.96
CELL_SIZE_X_M = None
CELL_SIZE_Y_M = None
TILE_COUNT_X = None
TILE_COUNT_Y = None
EXACT_CAD = True
REPAIR_CAD = False
REPAIR_BRIDGE_WIDTH_M = None
EXPORT_INSPECTION_SINGLE_FACE_STEP = False
SUBSTRATE_RHO = 7930.0
PIEZO_RHO = 7500.0


In [3]:
source_path = PROJECT_ROOT / SOURCE_UNIT_CELL_NPZ
if not source_path.exists():
    raise FileNotFoundError(
        f"Could not find {source_path}. Run periodic_grf_sdf.ipynb first or update SOURCE_UNIT_CELL_NPZ."
    )
print(f"Using source unit-cell dataset: {source_path}")

Using source unit-cell dataset: /home/gijeong/Inverse Design/data/unit_cell_dataset.npz


In [4]:
import importlib
import peh_inverse_design
import peh_inverse_design.pipeline_runner as pipeline_runner

importlib.reload(pipeline_runner)
importlib.reload(peh_inverse_design)

from peh_inverse_design import PipelineConfig, run_pipeline

config = PipelineConfig(
    source_unit_cell_npz=SOURCE_UNIT_CELL_NPZ,
    run_name=RUN_NAME,
    output_root=OUTPUT_ROOT,
    limit=LIMIT,
    sample_ids=SAMPLE_IDS,
    materialize_input_dataset=MATERIALIZE_INPUT_DATASET,
    problem_spec_path=PROBLEM_SPEC_PATH,
    docker_image=DOCKER_IMAGE,
    substrate_thickness_m=SUBSTRATE_THICKNESS_M,
    piezo_thickness_m=PIEZO_THICKNESS_M,
    mesh_size_scale=MESH_SIZE_SCALE,
    cad_reference_size_scale=CAD_REFERENCE_SIZE_SCALE,
    limit_solver_mesh_by_thickness=LIMIT_SOLVER_MESH_BY_THICKNESS,
    solver_mesh_backend=SOLVER_MESH_BACKEND,
    substrate_layers=SUBSTRATE_LAYERS,
    piezo_layers=PIEZO_LAYERS,
    solver_num_modes=SOLVER_NUM_MODES,
    solver_search_points=SOLVER_SEARCH_POINTS,
    peak_search_seed=PEAK_SEARCH_SEED,
    solver_element_order=SOLVER_ELEMENT_ORDER,
    strict_ansys_parity=STRICT_ANSYS_PARITY,
    solver_store_mode_shapes=SOLVER_STORE_MODE_SHAPES,
    skip_existing_solver_outputs=SKIP_EXISTING_SOLVER_OUTPUTS,
    cell_size_x_m=CELL_SIZE_X_M,
    cell_size_y_m=CELL_SIZE_Y_M,
    tile_count_x=TILE_COUNT_X,
    tile_count_y=TILE_COUNT_Y,
    substrate_rho=SUBSTRATE_RHO,
    piezo_rho=PIEZO_RHO,
    exact_cad=EXACT_CAD,
    repair_cad=REPAIR_CAD,
    repair_bridge_width_m=REPAIR_BRIDGE_WIDTH_M,
    export_inspection_single_face_step=EXPORT_INSPECTION_SINGLE_FACE_STEP,
    audit_ansys_modal_hz=AUDIT_ANSYS_MODAL_HZ,
    audit_ansys_frf_peak_hz=AUDIT_ANSYS_FRF_PEAK_HZ,
    audit_ansys_voltage_v=AUDIT_ANSYS_VOLTAGE_V,
    audit_ansys_voltage_form="peak",
)

artifacts = run_pipeline(config)
artifacts.as_dict()


Notebook/kernel python : /home/gijeong/miniconda/envs/GJ/bin/python
Pipeline step python   : /home/gijeong/Inverse Design/.venv/bin/python
Geometry scale       : cell=(0.1, 0.1) m, tiles=10 x 10, plate=(1, 1) m [source: problem_spec_used.yaml]
Run config snapshot : /home/gijeong/Inverse Design/runs/0409(5)/data/run_config_used.json
CAD export mode     : exact
Solver profile      : backend=layered_tet, mesh_preset=ansys_parity, layers=8+3, modes=8, search_points=301, peak_search_seed=f1, element_order=2, eigensolver_backend=shift_invert_lu, allow_eigensolver_fallback=False, strict_parity=True, skip_existing=True, mesh_size_scale=0.08, cad_reference_scale=0.01, max_q2_vector_dofs=None, oom_fallback_order=1, ansys_step_strategy=single_face_assembly, thickness_limited_mesh=False

Step 1/6: Export ANSYS STEP geometry and build Python solver meshes
$ '/home/gijeong/Inverse Design/.venv/bin/python' -m peh_inverse_design.build_volume_meshes --unit-cell-npz '/home/gijeong/Inverse Design/runs/04


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


[1/1] Solving plate3d_0000_fenicsx.npz
Saved solver diagnostic payload to /workspace/runs/0409(5)/data/fem_responses/sample_0000_solver_diagnostic.json


petsc4py.PETSc.Error: error code 76
[0] EPSSolve() at /usr/local/slepc/src/eps/interface/epssolve.c:132
[0] EPSSetUp() at /usr/local/slepc/src/eps/interface/epssetup.c:408
[0] STSetUp() at /usr/local/slepc/src/sys/classes/st/interface/stsolve.c:546
[0] STSetUp_Sinvert() at /usr/local/slepc/src/sys/classes/st/impls/sinvert/sinvert.c:191
[0] KSPSetUp() at /usr/local/petsc/src/ksp/ksp/interface/itfunc.c:429
[0] PCSetUp() at /usr/local/petsc/src/ksp/pc/interface/precon.c:1120
[0] PCSetUp_LU() at /usr/local/petsc/src/ksp/pc/impls/factor/lu/lu.c:121
[0] MatLUFactorNumeric() at /usr/local/petsc/src/mat/interface/matrix.c:3336
[0] MatFactorNumeric_MUMPS() at /usr/local/petsc/src/mat/impls/aij/mpi/mumps/mumps.c:2092
[0] Error in external library
[0] MUMPS error in numerical factorization: INFOG(1)=-13, INFO(2)=-54974 (see users manual https://mumps-solver.org/index.php?page=doc "Error and warning diagnostics")

The above exception was the direct cause of the following exception:

Traceback (mos

CalledProcessError: Command '['docker', 'run', '--rm', '-v', '/home/gijeong/Inverse Design:/workspace', '-w', '/workspace', 'dolfinx/dolfinx:stable', 'bash', '-c', "pip install -q pyyaml && python3 -m peh_inverse_design.fenicsx_modal_solver --mesh-dir '/workspace/runs/0409(5)/meshes/volumes' --response-dir '/workspace/runs/0409(5)/data/fem_responses' --modes-dir '/workspace/runs/0409(5)/data/modal_data' --substrate-rho 7930.0 --piezo-rho 7500.0 --num-modes 8 --search-points 301 --peak-search-seed f1 --element-order 2 --requested-element-order 2 --eigensolver-backend shift_invert_lu --strict-parity --store-mode-shapes --skip-existing --problem-spec '/workspace/runs/0409(5)/data/problem_spec_used.yaml'"]' returned non-zero exit status 1.

## Integrated Dataset Summary

In [ ]:
import numpy as np
from pathlib import Path


def _bool_at(array, idx):
    return bool(np.asarray(array[idx]).reshape(-1)[0])


def _float_at(array, idx):
    return float(np.asarray(array[idx], dtype=np.float64).reshape(-1)[0])


def _int_at(array, idx):
    return int(np.asarray(array[idx], dtype=np.int64).reshape(-1)[0])


def _str_at(array, idx):
    return str(np.asarray(array[idx]).reshape(-1)[0])


def _solver_mesh_backend(mesh_npz_path):
    path = Path(str(np.asarray(mesh_npz_path).reshape(-1)[0]))
    with np.load(path, allow_pickle=True) as mesh:
        if "solver_mesh_backend" not in mesh.files:
            return "unknown"
        return str(np.asarray(mesh["solver_mesh_backend"]).reshape(-1)[0])


data = np.load(artifacts.integrated_dataset_path, allow_pickle=True)
print("integrated dataset:", artifacts.integrated_dataset_path)
print("keys:")
print(sorted(data.files))
print()
print("sample summary")
for idx, sid in enumerate(data["sample_id"]):
    strict_parity_requested = _bool_at(data["strict_parity_requested"], idx)
    solver_parity_valid = _bool_at(data["solver_parity_valid"], idx)
    diagnostic_only = _bool_at(data["diagnostic_only"], idx)
    parity_invalid_reason = _str_at(data["parity_invalid_reason"], idx)
    if strict_parity_requested and (not solver_parity_valid or diagnostic_only):
        reason = parity_invalid_reason or "strict parity became invalid"
        raise RuntimeError(
            f"sample {int(sid):04d} requested strict parity but produced invalid/diagnostic outputs: {reason}"
        )
    print(f"sample_id={int(sid):04d}")
    print(f"  mesh_preset={_str_at(data['mesh_preset'], idx)}")
    print(f"  solver_mesh_backend={_solver_mesh_backend(data['mesh_npz_path'][idx])}")
    print(f"  substrate_layers={_int_at(data['substrate_layers'], idx)}")
    print(f"  piezo_layers={_int_at(data['piezo_layers'], idx)}")
    print(f"  strict_parity_requested={strict_parity_requested}")
    print(f"  solver_parity_valid={solver_parity_valid}")
    print(f"  diagnostic_only={diagnostic_only}")
    print(f"  frf_search_seed_source={_str_at(data['frf_search_seed_source'], idx)}")
    print(f"  frf_search_seed_frequency_hz={_float_at(data['frf_search_seed_frequency_hz'], idx):.6f}")
    print(f"  mode1_frequency_hz={_float_at(data['mode1_frequency_hz'], idx):.6f}")
    print(
        "  dominant_drive_coupling_mode_frequency_hz="
        f"{_float_at(data['dominant_drive_coupling_mode_frequency_hz'], idx):.6f}"
    )
    print(f"  f_peak_hz={_float_at(data['f_peak_hz'], idx):.6f}")
    print(f"  peak_voltage_peak_v={_float_at(data['peak_voltage_peak_v'], idx):.6f}")
    print()

## Report Images

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown(f"**Run root:** `{artifacts.run_root}`"))
display(Markdown(f"**Integrated dataset:** `{artifacts.integrated_dataset_path}`"))
display(Markdown(f"**Index CSV:** `{artifacts.integrated_index_csv_path}`"))

gallery_path = Path(artifacts.gallery_path)
if gallery_path.exists():
    display(Image(filename=str(gallery_path)))
else:
    print(f"Gallery image not found: {gallery_path}")

Outputs created by this notebook:

- `runs/<RUN_NAME>/meshes/volumes/`
- `runs/<RUN_NAME>/data/fem_responses/`
- `runs/<RUN_NAME>/data/modal_data/`
- `runs/<RUN_NAME>/data/response_dataset.npz`
- `runs/<RUN_NAME>/data/integrated_dataset.npz`
- `runs/<RUN_NAME>/data/integrated_dataset.csv`
- `runs/<RUN_NAME>/reports/`
